# Lab 1: NumPy for Cat and Dog Faces

Completed solution notebook.


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

LABELS = ("cat", "dog")
LABEL_TO_INDEX = {"cat": 0, "dog": 1}
CHANNEL_NAMES = np.array(["red", "green", "blue"])
EDGE_KERNEL = np.array(
    [
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0],
    ],
    dtype=np.float32,
)
FEATURE_NAMES = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b",
    "brightest_channel",
    "edge_mean",
    "edge_std",
    "row_std_mean",
]


def label_from_path(path: Path) -> str:
    label = path.parent.name
    if label not in LABEL_TO_INDEX:
        raise ValueError(f"Unexpected label folder: {path}")
    return label


def load_image_np(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))


def center_crop(image: np.ndarray, crop_size: int = 48) -> np.ndarray:
    height, width = image.shape[:2]
    top = (height - crop_size) // 2
    left = (width - crop_size) // 2
    return image[top:top + crop_size, left:left + crop_size, ...]


def flip_horizontal(image: np.ndarray) -> np.ndarray:
    return image[:, ::-1, ...].copy()


def normalize_01(image: np.ndarray) -> np.ndarray:
    return image.astype(np.float32) / 255.0


def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
    return image_float @ weights


def channel_summary(image_float: np.ndarray) -> tuple[np.ndarray, int]:
    channel_means = image_float.mean(axis=(0, 1)).astype(np.float32)
    brightest_channel = int(np.argmax(channel_means))
    return channel_means, brightest_channel


def convolve2d_matmul(image_gray: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    height, width = image_gray.shape
    kernel_height, kernel_width = kernel.shape

    output_height = height - kernel_height + 1
    output_width = width - kernel_width + 1

    output = np.empty((output_height, output_width), dtype=np.float32)
    kernel_flat = kernel.reshape(-1)

    for i in range(output_height):
        for j in range(output_width):
            patch = image_gray[i:i + kernel_height, j:j + kernel_width]
            output[i, j] = patch.reshape(-1) @ kernel_flat

    return output


def flatten_image(image: np.ndarray) -> np.ndarray:
    return image.reshape(-1)


def extract_features(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    cropped = center_crop(image, crop_size=48)
    image_float = normalize_01(cropped)
    gray = rgb_to_gray(image_float)

    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1)).astype(np.float32)

    filtered = convolve2d_matmul(gray, kernel)
    row_std_profile = np.apply_along_axis(np.std, 1, gray)
    row_std_mean = row_std_profile.mean().astype(np.float32)

    features = np.concatenate(
        [
            channel_means,
            channel_stds,
            np.array(
                [
                    float(brightest_channel),
                    float(filtered.mean()),
                    float(filtered.std()),
                    float(row_std_mean),
                ],
                dtype=np.float32,
            ),
        ]
    ).astype(np.float32)

    return features


def build_feature_matrix(paths: list[Path], kernel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    features = []
    labels = []

    for path in paths:
        image = load_image_np(path)
        features.append(extract_features(image, kernel))
        labels.append(LABEL_TO_INDEX[label_from_path(path)])

    X = np.stack(features).astype(np.float32)
    y = np.array(labels, dtype=np.int64)

    return X, y


## Reflection

1. Keeping the crop size fixed makes every image have the same shape before feature extraction. This is necessary because the feature vectors must have consistent dimensions.

2. `axis=(0, 1)` means NumPy computes across the height and width dimensions while keeping the color channel dimension. For an RGB image, this gives one mean value for red, green, and blue.

3. The edge filter captures changes in brightness between neighboring pixels. RGB means only describe average color, while the edge response gives information about texture, outlines, and local structure.

4. Flattening converts an image into one vector, which is useful for building feature matrices and vector-based operations. However, it weakens the original 2D spatial layout because pixel positions are no longer represented as rows and columns.
